In [1]:
import ollama
import pandas as pd
import numpy as np

En este notebook voy a aplicar todas las transformaciones necesarias para trabajar la variable `Descripción`, utilizando modelos de lenguaje natural en local.

# Glassdoor

In [2]:
df = pd.read_csv('../data/raw/glassdoor.csv')

In [3]:
df.head()

,Titulo,Empresa,Localizacion,Salario,Fecha_Publicacion,Valoracion_Empresa,Descripcion_Completa,Busqueda_Ciudad,Busqueda_Puesto,Sector
0,FISIOTERAPEUTA NEUROLÓGICO,Centro Monte del Pilar,Madrid,24 mil € - 26 mil € (Proporcionado por la empr...,24 horas,"3,9",Oferta de empleo – Fisioterapeuta Neurológico/...,Madrid,Todos,NaN
1,Comercial puerta fría (presencial) - Sector ba...,Total Management Consulting,Madrid,1 mil € - 6 mil € (Proporcionado por la empresa),24 horas,"3,9",Desde Total Management Consulting nos encontra...,Madrid,Todos,NaN
2,Profesora/a de Lengua 2ºESO (Aravaca),Verona Servicios de Intermediación Educativa S.L.,Madrid,"10,00 € - 12,00 € Por hora (Proporcionado por ...",24 horas,"3,9",Estamos seleccionando un profesor/a para impar...,Madrid,Todos,NaN
3,Direct Entry Captain - Pilot,The Emirates Group,Madrid,No especificado,+ 30 d,"3,9",Company Introduction\n\n\nFrom its global hub ...,Madrid,Todos,NaN
4,Promotor/a - Encuestador/a de Educación (Campu...,Cambridge Exams,Madrid,1 mil € (Proporcionado por la empresa),24 horas,"3,9","¿Te consideras una persona extrovertida, empát...",Madrid,Todos,NaN


En primer lugar, quiero sacar de la descripcion la siguiente información: 
- Tipo de empleo
- Modalidad
- Sector
- Nivel de experiencia
- Habilidades requeridas o competencias
- Beneficios 
 
Prueba con 1 registro:

In [3]:
sectores = {  #los he sacado de statista
    "Publicidad y Marketing": "Publicidad, Marcas y líderes, Marketing",
    "Agricultura": "Agricultura, Pesca y acuicultura, Silvicultura",
    "Productos químicos y recursos": "Industria química, Combustibles fósiles, Minería, metales y minerales, Petróleo y refinería, Plástico y caucho, Pulpa y papel",
    "Construcción": "Construcción de edificios, Construcción pesada",
    "Bienes de consumo": "Bebidas alcohólicas, Ropa y zapatos, Productos de limpieza, Cosméticos y cuidado personal, Alimentación y nutrición, Muebles, decoración y artículos para el hogar",
    "Comercio electrónico": "Comercio electrónico B2B, Comercio electrónico B2C, Comercio electrónico C2C, Comportamiento de compra digital, Cifras clave del comercio electrónico, Contenido de pago",
    "Economía y política": "Economía, Internacional, Política y Gobierno",
    "Energía y medio ambiente": "Clima y tiempo, Emisiones, Energía, Tecnología ambiental y tecnología verde, Gestión de residuos, Agua y aguas residuales",
    "Finanzas y seguros": "Instituciones financieras, Instrumentos financieros e inversiones, Servicios financieros, Seguro",
    "Salud, Farmacia y Tecnología Médica": "Cuidado y apoyo, Profesionales de la salud y hospitales, Tecnología médica, Productos farmacéuticos y mercado",
    "Internet": "Comunicaciones, Delitos cibernéticos y seguridad, Demografía y uso, Internet móvil y aplicaciones, Búsqueda en línea, Vídeo y entretenimiento en línea, Alcance y tráfico, Redes sociales y contenido generado por el usuario",
    "Medios de comunicación": "Audio, Libros y publicaciones, Noticias, Televisión, vídeo y cine, Videojuegos y deportes electrónicos",
    "Metales y electrónica": "Fabricación aeroespacial y de defensa, Electrónica, Fabricación de maquinaria industrial, Rieles, Fabricación de material rodante, Construcción naval, Fabricación de vehículos",
    "Bienes raíces": "Bienes raíces comerciales, Bienes raíces industriales, Hipotecas y Financiamiento, Servicios inmobiliarios, Bienes raíces residenciales",
    "Comercio minorista y comercio": "Venta minorista de bricolaje, Moda y accesorios, Alimentos y bebidas Venta al por menor de muebles Mercancía general Salud e higiene Comercio internacional Material de oficina Marca privada Tecnología minorista Comportamiento de compra, Deportes y ocio, Suscripciones y venta directa, Cadena de suministro, Al por mayor",
    "Servicios": "Servicios empresariales, Mano de obra calificada",
    "Sociedad": "Crimen y aplicación de la ley, Demografía, Educación y ciencia, Geografía y naturaleza, Datos históricos, Religión",
    "Deportes y recreación": "Arte y cultura, Juego, Aficiones, Parques y actividades al aire libre, Deportes profesionales, Deportes y fitness, Bienestar y spas",
    "Tecnología y telecomunicaciones": "Electrónica de consumo, Hardware, Electrodomésticos, Servicios de TI, Software, Telecomunicaciones",
    "Transporte y Logística": "Aviación, Logística, Servicios de transporte público y movilidad, Transporte ferroviario, Vehículos y tráfico rodado, Transporte acuático",
    "Viajes, turismo y hostelería": "Alojamiento, Viajes de negocios, Servicios de comida y bebida, Viajes de ocio"
}

CIUDADES = [
    "Madrid","Barcelona","Valencia","Sevilla","Zaragoza","Málaga",
    "Murcia","Bilbao","Alicante","Valladolid","Palma de Mallorca",
    "Las Palmas de Gran Canaria","San Sebastián","A Coruña","Granada",
    "Vigo","Córdoba","Santander","Pamplona","Oviedo","Gijón",
    "Vitoria","Santa Cruz de Tenerife","Salamanca","Burgos",
    "Cádiz","Tarragona","León","Jerez de la Frontera","Castellón de la Plana",
    "Albacete","Almería","Ávila","Badajoz","Cáceres","Ciudad Real",
    "Cuenca","Girona","Guadalajara","Huelva","Huesca",
    "Jaén","Logroño","Lleida","Lugo","Ourense","Palencia",
    "Pontevedra","Segovia","Soria","Teruel","Toledo","Zamora"
]

In [4]:
prompt = f"""
Eres un experto en Data Science aplicado a Recursos Humanos. Tu tarea es procesar ofertas de empleo y devolver un JSON perfectamente estructurado.

### REGLAS CRÍTICAS
1. IDIOMA: Extrae y traduce todo al español.
2. NULOS: Si un dato no existe, usa obligatoriamente 'null' (sin comillas para números, con comillas para strings si el esquema lo pide), excepto para formación_académica.
3. FORMATO: Devuelve ÚNICAMENTE el objeto JSON, sin texto explicativo antes o después.
4. SALARIO: Si la oferta indica un sueldo mensual (ej. 2.000€/mes), multiplícalo por 12 y devuelve 24000.0.

### ESQUEMA Y LÓGICA DE CAMPOS (ESTRICTO Y REQUERIDO TODOS LOS CAMPOS)
- "años_experiencia": Entero. (Si dice 'meses' y es < 12, pon 0. Si no hay, null).
- "experiencia": Categoría única ['intern', 'junior', 'senior', null]. 
    * Prioridad 1: Lo que diga la oferta. 
    * Prioridad 2: Si años=0 -> 'intern'; si años 1-3 -> 'junior'; si años >3 -> 'senior'.
- "modalidad": Categoría única ['Remoto', 'Presencial', 'Hibrido', null].
- "habilidades": String único con las skills separadas por comas.
- "formación_académica": Categoría única ['Ninguna', 'ESO', 'FP Medio', 'FP Superior', 'Grado Universitario', 'Postgrado']. 
    * Si piden varias, elige la de NIVEL INFERIOR. Si no hay, 'Ninguna'.
- "beneficios": String único con los beneficios (el salario no está incluido aqui) o null.
- "salario_minimo": Si solo hay una cifra (ej. 30.000), úsala como mínimo y máximo. Si no hay, null.
- "salario_maximo": (Igual que el anterior).
- "sector": Elegir OBLIGATORIAMENTE una de las posibilidades
- "tipo_de_empleo": Categoría única ['Jornada completa', 'Media jornada', 'Temporal', 'Contrato por obra', 'Prácticas', 'Voluntariado', 'Otro', null].
- "ciudad": Si una ciudad no está en la lista permitida del esquema, mapeala a su capital de provincia, sino aparece nada pon null.

### EJEMPLO DE SALIDA
{{
    "años_experiencia": 1,
    "experiencia": "junior",
    "modalidad": "Hibrido",
    "habilidades": "Python, SQL, Análisis de datos, trabajo en equipo, formación en fisioterapia neurológica, habilidades de comunicación y empatía ",
    "formación_académica": "Grado Universitario",
    "beneficios": "Seguro médico, fruta en la oficina",
    "salario_minimo": 22000.0,
    "salario_maximo": 25000.0,
    "sector": "Tecnología y telecomunicaciones",
    "tipo_de_empleo": "Jornada completa",
    "ciudad": "Salamanaca"
}}
"""

Haciendo pruebas, me di cuenta que el modelo a veces "alucina", y es mucho más seguro crear una estructura de json fija que debe seguir.

In [5]:
job_schema = {
    "type": "object",
    "properties": {
        "años_experiencia": {"type": ["integer", "null"]},
        "experiencia": {"type": ["string", "null"], "enum": ["intern", "junior", "senior", None]},
        "modalidad": {"type": ["string", "null"], "enum": ["Remoto", "Presencial", "Hibrido", None]},
        "habilidades": {"type": "string"},
        "formación_académica": {
            "type": "string", 
            "enum": ["Ninguna", "ESO", "FP Medio", "FP Superior", "Grado Universitario", "Postgrado"]
        },
        "beneficios": {"type": ["string", "null"]},
        "salario_minimo": {"type": ["number", "null"]},
        "salario_maximo": {"type": ["number", "null"]},
        "sector": {"type": "string", "enum": list(sectores.keys())},
        "tipo_de_empleo": {
            "type": ["string", "null"], 
            "enum": ["Jornada completa", "Media jornada", "Temporal", "Contrato por obra", "Prácticas", "Voluntariado", "Otro", None]
        },
        "ciudad": {"type": ["string", "null"], "enum": CIUDADES + [None]}
    },
    "required": [
        "años_experiencia", "experiencia", "modalidad", "habilidades", 
        "formación_académica", "salario_minimo", "salario_maximo", "sector", "ciudad"
    ]
}

In [8]:

oferta_ejemplo = df.loc[133,'Descripcion_Completa']

response = ollama.chat(
    model = 'qwen2.5:14b-instruct',
    messages=[{'role':'system', 'content': prompt},
              {'role':'user', 'content': oferta_ejemplo}],
    format=job_schema,
    options={'temperature': 0.0,
             'num_ctx': 2048,
             'num_predict': 300,
             'low_vram': False}
)

print(response['message']['content'])


{
    "años_experiencia": 1,
    "experiencia": "junior",
    "modalidad": "Presencial",
    "habilidades": "Manejo de carretillas, Carnet de carretillas en vigor",
    "formación_académica": "Ninguna",
    "salario_minimo": 16000.0,
    "salario_maximo": 17000.0,
    "sector": "Servicios",
    "ciudad": null,
    "tipo_de_empleo": "Temporal"
}


Procesar todas las ofertas con el LLM en local es un proceso muy largo (unas 50 horas), por eso voy a usar JSONL. JSONL, frente a JSON o csv, permite tratar cada registro como una línea independiente y completa, lo que garantiza que si el script falla (o lo paro) en la oferta 5.000, las anteriores han quedado guardadas. Además me permite volver a comenzar en ese punto e ir procesando las descripciones en varios días.

In [7]:
import os
import time
from tqdm import tqdm
import json

PATH_DESTINO = '../data/interim/extracion_descripcion.jsonl'

# Lo primero es comprobar porque oferta va el proceso
inicio = 0
if os.path.exists(PATH_DESTINO): #si ya existe el archivo, se encarga de comprobar por que registro va
    with open(PATH_DESTINO, 'r', encoding='utf8') as f:
        inicio = sum(1 for _ in f) #esto será la suma de registros del json

print(f'Total ofertas: {df.shape[0]}')
print(f'Ya procesadas: {inicio}')

# Ahora viene el bucle para procesar
with open(PATH_DESTINO, 'a', encoding='utf-8') as f: #abrir el archivo en modo append, para no sobreescribir sino seguir añadiendo
    for i, row in tqdm(df.iloc[inicio:].iterrows(), total=df.shape[0], initial = inicio):
        # tqdm es una una biblioteca que permite crear barras de progreso [elemento a recorrer, cuantos pasos tiene el progreso, cual es el inicio], permite ver el progreso global
        # iterrrows permite que en cada vuelta del bucle i sea el índice de la fila y row los datos de la fila

        if i > 0 and i % 1000 == 0: #esto hace que cada mil registros se haga una pausa, para que la tarjeta grafica no se sobrecaliente
            print('Pausa')
            time.sleep(300)
        
        descripcion = str(row['Descripcion_Completa'])

        # Aplicar modelo
        try:
            response = ollama.chat(
                model = 'qwen2.5:14b-instruct',
                messages=[{'role':'system', 'content': prompt},
                          {'role':'user', 'content': f'Oferta: {descripcion}'}],
                format=job_schema,
                options={'temperature': 0.0,
                        'num_ctx': 2048,
                        'num_predict': 1300,
                        'low_vram': False}
            )

            respuesta = response['message']['content']
            respuesta_dict = json.loads(respuesta) #transformar string en diccionario

            registro_final = {
                "id_original": i,
                "datos": respuesta_dict
            }

            #Guardarlo
            f.write(json.dumps(registro_final, ensure_ascii=False) + '\n') #json.dumps convierte diccionario en json, el esnure ascii permite que halla ñ y se añade al final un salto de linea
            f.flush()
        except Exception as e:
            print(f"\nError procesando fila {i}: {e}")
            continue

Total ofertas: 21069
Ya procesadas: 20968


100%|█████████▉| 21000/21069 [05:25<10:06,  8.80s/it]  

Pausa


100%|██████████| 21069/21069 [20:02<00:00, 11.90s/it]  


In [8]:
prueba = pd.read_json('../data/interim/extracion_descripcion.jsonl', lines=True)

In [9]:
df_prueba = prueba.datos.apply(pd.Series)

In [10]:
df_prueba['salario_minimo'].notnull().sum()

np.int64(5047)

# Adzuna API

In [3]:
df_adzuna = pd.read_csv('../data/raw/AdzunaAPI.csv')

C:\Users\ivanb\AppData\Local\Temp\ipykernel_39248\1314792321.py:1: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df_adzuna = pd.read_csv('../data/raw/AdzunaAPI.csv')


In [9]:
df_adzuna.head()

,redirect_url,created,description,__CLASS__,salary_max,id,category,title,adref,location,salary_min,company,salary_is_predicted,latitude,longitude,contract_time,contract_type
0,https://www.adzuna.es/details/5477402347?utm_m...,2025-11-02T20:34:24Z,Ofrecemos: empleo bajo condiciones alemanas - ...,Adzuna::API::Response::Job,27600.0,5477402347,"{'tag': 'unknown', '__CLASS__': 'Adzuna::API::...",Cerrajero,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTQ3NzQwMjM0NyIsI...,"{'display_name': 'España', '__CLASS__': 'Adzun...",27600.0,{'__CLASS__': 'Adzuna::API::Response::Company'},0,NaN,NaN,NaN,NaN
1,https://www.adzuna.es/details/5477521584?utm_m...,2025-11-03T00:47:18Z,Te beneficiarás de: contrato laboral holandés ...,Adzuna::API::Response::Job,7560.0,5477521584,{'__CLASS__': 'Adzuna::API::Response::Category...,Instalador de sistemas sanitarios y de calefac...,eyJhbGciOiJIUzI1NiJ9.eyJzIjoibkk3U01TUFU4Qkc4U...,"{'area': ['España'], '__CLASS__': 'Adzuna::API...",7200.0,{'__CLASS__': 'Adzuna::API::Response::Company'},0,NaN,NaN,NaN,NaN
2,https://www.adzuna.es/details/5477402236?utm_m...,2025-11-02T20:34:23Z,Ofrecemos: empleo bajo condiciones belgas – co...,Adzuna::API::Response::Job,47604.0,5477402236,"{'label': 'Unknown', 'tag': 'unknown', '__CLAS...",Conductor CE (Bélgica),eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTQ3NzQwMjIzNiIsI...,"{'area': ['España'], 'display_name': 'España',...",47604.0,{'__CLASS__': 'Adzuna::API::Response::Company'},0,NaN,NaN,NaN,NaN
3,https://www.adzuna.es/details/5501685320?utm_m...,2025-11-18T13:24:29Z,Company: Gridlines Gridlines is a rapidly grow...,Adzuna::API::Response::Job,NaN,5501685320,"{'label': 'Otros trabajos', '__CLASS__': 'Adzu...",Senior Analyst- Model Audit,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTUwMTY4NTMyMCIsI...,{'__CLASS__': 'Adzuna::API::Response::Location...,NaN,{'__CLASS__': 'Adzuna::API::Response::Company'...,0,NaN,NaN,NaN,NaN
4,https://www.adzuna.es/details/5477401159?utm_m...,2025-11-02T20:34:10Z,Ofrecemos: contrato de trabajo austriaco - rel...,Adzuna::API::Response::Job,33600.0,5477401159,"{'label': 'Unknown', '__CLASS__': 'Adzuna::API...",MAG soldador para vehículos blindados,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTQ3NzQwMTE1OSIsI...,{'__CLASS__': 'Adzuna::API::Response::Location...,33600.0,{'__CLASS__': 'Adzuna::API::Response::Company'},0,NaN,NaN,NaN,NaN


In [10]:
df_adzuna['contract_type'].value_counts()

contract_type
permanent    4
contract     3
Name: count, dtype: int64

In [11]:
df_adzuna['contract_time'].value_counts()

contract_time
full_time    712
part_time      1
Name: count, dtype: int64

In [15]:
df_adzuna.loc[50,'category']

"{'label': 'Otros trabajos', '__CLASS__': 'Adzuna::API::Response::Category', 'tag': 'other-general-jobs'}"

In [4]:
df_adzuna.shape

(136400, 17)

In [5]:
df_adzuna_salary = df_adzuna[df_adzuna.salary_max.notnull()]

In [6]:
df_adzuna_salary.shape

(24525, 17)

Voy aplicar un muestreo aleatorio simple para quedarme con 15.000 ofertas, y junto a las 5.000 de glassdoor obtener en total 20.000 ofertas de trabajo con salario.

In [7]:
muestra = df_adzuna_salary.sample(n=15000, random_state=42)

In [8]:
muestra.shape

(15000, 17)

In [9]:
muestra.reset_index(drop=True, inplace=True)
muestra.head()

,redirect_url,created,description,__CLASS__,salary_max,id,category,title,adref,location,salary_min,company,salary_is_predicted,latitude,longitude,contract_time,contract_type
0,https://www.adzuna.es/land/ad/5512378982?se=wo...,2025-11-25T12:53:47Z,Una consultora de ingeniería y transporte en M...,Adzuna::API::Response::Job,125000.0,5512378982,"{'label': 'Trabajos en consultoría', '__CLASS_...",Líder de Proyectos Ferroviarios - Infraestruct...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTUxMjM3ODk4MiIsI...,"{'area': ['España', 'Comunidad de Madrid', 'Ma...",110000.0,"{'display_name': 'INECO', '__CLASS__': 'Adzuna...",0,40.4167,3.70325,NaN,NaN
1,https://www.adzuna.es/land/ad/5521450206?se=SF...,2025-12-01T13:01:00Z,Software Engineer Manager | Java & AWS | 100% ...,Adzuna::API::Response::Job,110000.0,5521450206,"{'tag': 'it-jobs', 'label': 'Trabajos en infor...",Software Engineer Manager | Java & AWS | 100% ...,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiU0ZkVkFTclU4QkdNO...,"{'area': ['España', 'Comunidad de Madrid', 'Ma...",90000.0,{'__CLASS__': 'Adzuna::API::Response::Company'...,0,40.4167,3.70325,NaN,NaN
2,https://www.adzuna.es/land/ad/5530884914?se=cp...,2025-12-06T14:51:00Z,A dynamic product company in Barcelona is look...,Adzuna::API::Response::Job,110000.0,5530884914,"{'label': 'Trabajos en informática', 'tag': 'i...",Senior Solution Architect – AdTech & Data Plat...,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiY3B4ZWVTX1U4QkduR...,"{'display_name': 'Barcelona, Cataluña', '__CLA...",90000.0,"{'display_name': 'Appodeal, Inc.', '__CLASS__'...",0,41.3851,2.17340,NaN,NaN
3,https://www.adzuna.es/details/5350502401?utm_m...,2025-08-13T15:25:15Z,"Coordinación de contratos, proveedores y presu...",Adzuna::API::Response::Job,50000.0,5350502401,"{'label': 'Trabajos en ingeniería', 'tag': 'en...",Jefe/a de Mantenimiento,eyJhbGciOiJIUzI1NiJ9.eyJzIjoicEd3QTR5YlU4QkdWd...,"{'area': ['España'], '__CLASS__': 'Adzuna::API...",45000.0,{'__CLASS__': 'Adzuna::API::Response::Company'},0,NaN,NaN,NaN,NaN
4,https://www.adzuna.es/details/5291390409?utm_m...,2025-07-08T07:47:20Z,"Gestionar y mantener la infraestructura IT, as...",Adzuna::API::Response::Job,35000.0,5291390409,"{'tag': 'it-jobs', '__CLASS__': 'Adzuna::API::...",Técnico de Infraestructuras IT,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTI5MTM5MDQwOSIsI...,"{'display_name': 'España', 'area': ['España'],...",28000.0,{'__CLASS__': 'Adzuna::API::Response::Company'},0,NaN,NaN,NaN,NaN


In [10]:
muestra.to_csv('../data/raw/AdzunaAPI_muestra.csv', index=False)

In [8]:
prompt = f"""
Eres un experto en Data Science aplicado a Recursos Humanos. Tu tarea es procesar ofertas de empleo y devolver un JSON perfectamente estructurado.

### REGLAS CRÍTICAS
1. IDIOMA: Extrae y traduce todo al español.
2. NULOS: Si un dato no existe, usa obligatoriamente 'null' (sin comillas para números, con comillas para strings si el esquema lo pide), excepto para formación_académica.
3. FORMATO: Devuelve ÚNICAMENTE el objeto JSON, sin texto explicativo antes o después.

### ESQUEMA Y LÓGICA DE CAMPOS (ESTRICTO Y REQUERIDO TODOS LOS CAMPOS)
- "años_experiencia": Entero. (Si dice 'meses' y es < 12, pon 0. Si no hay, null).
- "experiencia": Categoría única ['intern', 'junior', 'senior', null]. 
    * Prioridad 1: Lo que diga la oferta. 
    * Prioridad 2: Si años=0 -> 'intern'; si años 1-3 -> 'junior'; si años >3 -> 'senior'.
- "modalidad": Categoría única ['Remoto', 'Presencial', 'Hibrido', null].
- "habilidades": String único con las skills separadas por comas.
- "formación_académica": Categoría única ['Ninguna', 'ESO', 'FP Medio', 'FP Superior', 'Grado Universitario', 'Postgrado']. 
    * Si piden varias, elige la de NIVEL INFERIOR. Si no hay, 'Ninguna'.
- "beneficios": String único con los beneficios (el salario no está incluido aqui) o null.
- "sector": Elegir OBLIGATORIAMENTE una de las posibilidades
- "tipo_de_empleo": Categoría única ['Jornada completa', 'Media jornada', 'Temporal', 'Contrato por obra', 'Prácticas', 'Voluntariado', 'Otro', null].
- "ciudad": Si una ciudad no está en la lista permitida del esquema, mapeala a su capital de provincia, sino aparece nada pon null.

### EJEMPLO DE SALIDA
{{
    "años_experiencia": 1,
    "experiencia": "junior",
    "modalidad": "Hibrido",
    "habilidades": "Python, SQL, Análisis de datos, trabajo en equipo, formación en fisioterapia neurológica, habilidades de comunicación y empatía ",
    "formación_académica": "Grado Universitario",
    "beneficios": "Seguro médico, fruta en la oficina",
    "sector": "Tecnología y telecomunicaciones",
    "tipo_de_empleo": "Jornada completa",
    "ciudad": "Salamanaca"
}}
"""

job_schema = {
    "type": "object",
    "properties": {
        "años_experiencia": {"type": ["integer", "null"]},
        "experiencia": {"type": ["string", "null"], "enum": ["intern", "junior", "senior", None]},
        "modalidad": {"type": ["string", "null"], "enum": ["Remoto", "Presencial", "Hibrido", None]},
        "habilidades": {"type": "string"},
        "formación_académica": {
            "type": "string", 
            "enum": ["Ninguna", "ESO", "FP Medio", "FP Superior", "Grado Universitario", "Postgrado"]
        },
        "beneficios": {"type": ["string", "null"]},
        "sector": {"type": "string", "enum": list(sectores.keys())},
        "tipo_de_empleo": {
            "type": ["string", "null"], 
            "enum": ["Jornada completa", "Media jornada", "Temporal", "Contrato por obra", "Prácticas", "Voluntariado", "Otro", None]
        },
        "ciudad": {"type": ["string", "null"], "enum": CIUDADES + [None]}
    },
    "required": [
        "años_experiencia", "experiencia", "modalidad", "habilidades", 
        "formación_académica", "sector", "ciudad"
    ]
}

In [9]:
import os
import time
from tqdm import tqdm
import json

PATH_DESTINO = '../data/interim/extracion_descripcion_adzuna.jsonl'

# Lo primero es comprobar porque oferta va el proceso
inicio = 0
if os.path.exists(PATH_DESTINO): #si ya existe el archivo, se encarga de comprobar por que registro va
    with open(PATH_DESTINO, 'r', encoding='utf8') as f:
        inicio = sum(1 for _ in f) #esto será la suma de registros del json

print(f'Total ofertas: {muestra.shape[0]}')
print(f'Ya procesadas: {inicio}')

# Ahora viene el bucle para procesar
with open(PATH_DESTINO, 'a', encoding='utf-8') as f: #abrir el archivo en modo append, para no sobreescribir sino seguir añadiendo
    for i, row in tqdm(muestra.iloc[inicio:].iterrows(), total=muestra.shape[0], initial = inicio):
        # tqdm es una una biblioteca que permite crear barras de progreso [elemento a recorrer, cuantos pasos tiene el progreso, cual es el inicio], permite ver el progreso global
        # iterrrows permite que en cada vuelta del bucle i sea el índice de la fila y row los datos de la fila

        if i > 0 and i % 1000 == 0: #esto hace que cada mil registros se haga una pausa, para que la tarjeta grafica no se sobrecaliente
            print('Pausa')
            time.sleep(300)
        
        descripcion = str(row['description']) + " " + str(row['category']) + str(row['location'])

        # Aplicar modelo
        try:
            response = ollama.chat(
                model = 'qwen2.5:14b-instruct',
                messages=[{'role':'system', 'content': prompt},
                          {'role':'user', 'content': f'Oferta: {descripcion}'}],
                format=job_schema,
                options={'temperature': 0.0,
                        'num_ctx': 2048,
                        'num_predict': 1300,
                        'low_vram': False}
            )

            respuesta = response['message']['content']
            respuesta_dict = json.loads(respuesta) #transformar string en diccionario

            registro_final = {
                "id_original": i,
                "datos": respuesta_dict
            }

            #Guardarlo
            f.write(json.dumps(registro_final, ensure_ascii=False) + '\n') #json.dumps convierte diccionario en json, el esnure ascii permite que halla ñ y se añade al final un salto de linea
            f.flush()
        except Exception as e:
            print(f"\nError procesando fila {i}: {e}")
            continue

Total ofertas: 15000
Ya procesadas: 14216


100%|██████████| 15000/15000 [1:18:08<00:00,  5.98s/it]
